# 9-Qubit Shor Code in MLIR (Python bindings)

This notebook builds an **MLIR module** for the 9-qubit Shor encode/decode circuit using **Python MLIR bindings** (no full LLVM build).

We keep the IR portable by representing quantum gates as **external function calls** (`@h`, `@cx`, etc.) inside standard MLIR dialects (`func`, `tensor`, `arith`).

You can later swap these externs for a real quantum runtime / dialect lowering.


In [ ]:
# If you opened this notebook without installing deps, uncomment: 
# %pip -q install -r ../requirements.txt

import numpy as np

from src.shor_mlir import build_shor_module, canonicalize_and_cse


## 1) Build MLIR for Shor encode/decode

We construct a module with:
- `func.func private @h(%q:i1)->i1`
- `func.func private @cx(%c:i1,%t:i1)->(i1,i1)`
- `func.func @shor_encode(%psi:i1)->tensor<9xi1>`
- `func.func @shor_decode(%code:tensor<9xi1>)->i1`

Here `i1` is just a placeholder type for a qubit handle; the goal is to get a clean MLIR graph you can later lower/translate.


In [ ]:
m = build_shor_module()
print(m)


## 2) Run a small pass pipeline

We run `canonicalize`, `cse`, and `symbol-dce` just to show the standard Python pass manager flow.


In [ ]:
canonicalize_and_cse(m)
print(m)


## 3) (Optional) Sanity check with a tiny NumPy simulator

This is **not** executing the MLIR. It just verifies the **encode/decode structure** matches the usual Shor circuit (encode then inverse-decode = identity on the logical qubit when no error is applied).

We simulate gates directly on a 9-qubit statevector (dimension 512), using the same gate order as in the MLIR builder.


In [ ]:
def kron_n(*ops):
    out = ops[0]
    for op in ops[1:]:
        out = np.kron(out, op)
    return out

I = np.eye(2, dtype=complex)
X = np.array([[0,1],[1,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)
H = (1/np.sqrt(2)) * np.array([[1,1],[1,-1]], dtype=complex)

def apply_1q(state, U, q, n=9):
    # q=0 is the most-significant qubit in this convention
    ops = [I]*n
    ops[q] = U
    Ufull = kron_n(*ops)
    return Ufull @ state

def apply_cx(state, c, t, n=9):
    # Build CX as projector method
    P0 = np.array([[1,0],[0,0]], dtype=complex)
    P1 = np.array([[0,0],[0,1]], dtype=complex)
    ops0 = [I]*n
    ops1 = [I]*n
    ops0[c] = P0
    ops1[c] = P1
    U0 = kron_n(*ops0)
    ops1[t] = X
    U1 = kron_n(*ops1)
    return (U0 + U1) @ state

def shor_encode_state(state):
    # same sequence as MLIR builder
    state = apply_cx(state, 0, 3)
    state = apply_cx(state, 0, 6)
    for q in (0,3,6):
        state = apply_1q(state, H, q)
    state = apply_cx(state, 0, 1)
    state = apply_cx(state, 0, 2)
    state = apply_cx(state, 3, 4)
    state = apply_cx(state, 3, 5)
    state = apply_cx(state, 6, 7)
    state = apply_cx(state, 6, 8)
    return state

def shor_decode_state(state):
    # inverse sequence
    state = apply_cx(state, 6, 8)
    state = apply_cx(state, 6, 7)
    state = apply_cx(state, 3, 5)
    state = apply_cx(state, 3, 4)
    state = apply_cx(state, 0, 2)
    state = apply_cx(state, 0, 1)
    for q in (0,3,6):
        state = apply_1q(state, H, q)
    state = apply_cx(state, 0, 6)
    state = apply_cx(state, 0, 3)
    return state

def basis(n, bits_int):
    v = np.zeros((2**n,), dtype=complex)
    v[bits_int] = 1.0
    return v

# Prepare |psi> on q0 and |0...0> on others.
# We'll test with |0> and |1>, and a random superposition.
zero9 = basis(9, 0)

def init_logical(alpha, beta):
    # state = (alpha|0> + beta|1>) \otimes |0..0>
    # With q0 as MSB, |1> on q0 corresponds to basis index 2^(8) = 256
    return alpha * basis(9, 0) + beta * basis(9, 256)

tests = [
    (1.0+0j, 0.0+0j),
    (0.0+0j, 1.0+0j),
    (0.37+0.12j, 0.81-0.44j),
]

for a,b in tests:
    # normalize
    nrm = np.sqrt(abs(a)**2 + abs(b)**2)
    a, b = a/nrm, b/nrm

    s0 = init_logical(a,b)
    s1 = shor_encode_state(s0)
    s2 = shor_decode_state(s1)

    # Compare reduced logical amplitudes (q0) by reading the |0..0> and |1 0..0> components.
    a2 = s2[0]
    b2 = s2[256]
    phase = a2/a if abs(a) > 1e-9 else b2/b

    ok = np.allclose(a2, phase*a, atol=1e-8) and np.allclose(b2, phase*b, atol=1e-8)
    print('PASS' if ok else 'FAIL', ' | recovered up to global phase')


## 4) Where to go next

To turn this into a *real* compiler path, you can add: 
- A proper `qubit` type (or use a quantum dialect if you later depend on CIRCT/QIR/other tooling)
- Lowerings from `@h/@cx` extern calls into your target runtime IR
- Error insertion passes (X/Z on selected wires)
- Syndrome extraction + correction (measurement + classical control), represented either via calls or via a dedicated dialect
